# Repeated subject-level partitions

In [ ]:
import os
try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as e:
    print("Not on Colab or already mounted:", e)

In [ ]:
PROJECT        = "/content/drive/MyDrive/Depression_FL_Project"
LABELS_CSV     = f"{PROJECT}/fl/labels_v1/fl_harmonized_labels.csv"
EEG_IDS_DIR    = f"{PROJECT}/data/embeddings"          # eeg_subj_ids_{train,val,test}.npy (ids only)
TEXT_EMBED_NPZ = f"{PROJECT}/output/PDCH/processed_v1/embeddings/pdch_text_emb_full_dialogue.npz"
AUDIO_TURN_DIR = f"{PROJECT}/audio/v1/turn_embeddings"

R1_OUT         = f"{PROJECT}/fl/revision_r1"
N_PARTITIONS   = 10          # number of independent subject-level partitions (R)
VAL_FRAC, TEST_FRAC = 0.20, 0.20
BASE_SEED      = 1000        # partition r uses random_state = BASE_SEED + r
os.makedirs(R1_OUT, exist_ok=True)
for p in (LABELS_CSV, EEG_IDS_DIR, TEXT_EMBED_NPZ, AUDIO_TURN_DIR):
    print(f"  exists={os.path.exists(p)}  {p}")

In [ ]:
import glob, json
import numpy as np, pandas as pd
from sklearn.model_selection import train_test_split

labels = pd.read_csv(LABELS_CSV)
labels["subject_id"] = labels["subject_id"].astype(str)

def ids_eeg():
    ids = [np.load(f"{EEG_IDS_DIR}/eeg_subj_ids_{s}.npy", allow_pickle=True) for s in ("train", "val", "test")]
    return set(str(i) for i in np.concatenate(ids))

def ids_text():
    return set(str(i) for i in np.load(TEXT_EMBED_NPZ, allow_pickle=True)["ids"])

def ids_audio():
    return set(str(int(os.path.basename(f).split(".")[0])) for f in glob.glob(f"{AUDIO_TURN_DIR}/*.npz"))

available = {"eeg": ids_eeg(), "text": ids_text(), "audio": ids_audio()}

def silo_table(silo):
    sub = labels[(labels.silo == silo) & labels.y_unified.notna()].copy()
    sub = sub[sub.subject_id.isin(available[silo])]
    sub["y_unified"] = sub["y_unified"].astype(int)
    return sub[["subject_id", "y_unified"]].sort_values("subject_id").reset_index(drop=True)

tables = {s: silo_table(s) for s in ("eeg", "text", "audio")}
for s, t in tables.items():
    print(f"{s:6s} n={len(t):4d}  class counts (none/mild-mod/severe) = {np.bincount(t.y_unified, minlength=3).tolist()}")

In [ ]:
def one_partition(df, seed):
    idx = np.arange(len(df)); y = df.y_unified.to_numpy()
    tr, rest = train_test_split(idx, test_size=VAL_FRAC + TEST_FRAC, random_state=seed, stratify=y)
    va, te = train_test_split(rest, test_size=TEST_FRAC / (VAL_FRAC + TEST_FRAC),
                              random_state=seed, stratify=y[rest])
    split = np.empty(len(df), dtype=object)
    split[tr], split[va], split[te] = "train", "val", "test"
    return split

rows = []
for silo, df in tables.items():
    for r in range(N_PARTITIONS):
        sp = one_partition(df, BASE_SEED + r)
        for sid, y, s in zip(df.subject_id, df.y_unified, sp):
            rows.append({"silo": silo, "partition": r, "subject_id": sid, "y_unified": int(y), "split": s})
parts = pd.DataFrame(rows)

chk = parts.groupby(["silo", "partition", "subject_id"]).size()
assert (chk == 1).all(), "a subject appears in more than one split of the same partition"
parts.to_csv(f"{R1_OUT}/partitions.csv", index=False)
print("saved ->", f"{R1_OUT}/partitions.csv", "|", len(parts), "rows")

In [ ]:
cnt = (parts.groupby(["silo", "partition", "split", "y_unified"]).size()
            .unstack(fill_value=0).reindex(columns=[0, 1, 2], fill_value=0))
mean_cnt = cnt.groupby(["silo", "split"]).mean().round(1)
print(mean_cnt)

order = [("eeg", "EEG (MODMA)"), ("audio", "Audio (DAIC-WOZ)"), ("text", "Text (PDCH)")]
lines = [r"\begin{tabular}{llcccc}", r"\toprule",
         r"\textbf{Silo} & \textbf{Split} & \textbf{None-min.} & \textbf{Mild-mod.} & \textbf{Severe} & \textbf{Total} \\",
         r"\midrule"]
for silo, nm in order:
    for k, sp in enumerate(("train", "val", "test")):
        v = mean_cnt.loc[(silo, sp)].to_numpy()
        lines.append(f"{nm if k == 0 else ''} & {sp} & {v[0]:g} & {v[1]:g} & {v[2]:g} & {v.sum():g} \\\\")
    lines.append(r"\midrule" if silo != "text" else r"\bottomrule")
lines.append(r"\end{tabular}")
os.makedirs(f"{R1_OUT}/latex", exist_ok=True)
open(f"{R1_OUT}/latex/tab_splitcounts.tex", "w").write("\n".join(lines))
print("saved ->", f"{R1_OUT}/latex/tab_splitcounts.tex")
json.dump({"N_PARTITIONS": N_PARTITIONS, "VAL_FRAC": VAL_FRAC, "TEST_FRAC": TEST_FRAC,
           "BASE_SEED": BASE_SEED, "stratify_on": "y_unified (3-class)"},
          open(f"{R1_OUT}/partitions_config.json", "w"), indent=2)